# 고정 M2를 M3·M4-C에 추가하는 2시드 2×2 개발 게이트

M2를 다시 설계하지 않고, 기존 개인이력 N/V M2(`axis_dim=4`, `rho=.05`)가 현재 M3+M4-C에 실제 추가효과를 주는지 확인합니다.

- 재사용: seed42·44의 M1, M3, M3+M4-C
- 신규학습: seed별 M2+M3와 M2+M3+M4-C, 총 4회
- 고정: Dunnhumby DAY1~683 학습, DAY684~690 개발평가, 300 epoch 마지막점 판정
- 학습: 각 arm을 무작위 초기화부터 하나의 forward·optimizer로 공동학습
- 금지: 사전학습·동결·외부 점수합산·재정렬·최종주·holdout·best epoch 선택

주 비교는 `(M2+M3+M4-C) - (M3+M4-C)`입니다. 두 경제지표@10 평균 상승, 여섯 Recall/NDCG의 99% 보호, 두 시드 방향을 함께 확인합니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, subprocess, sys

if not os.path.ismount('/content/drive'):
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패. 학습을 시작하지 않았습니다.') from exc

ROOT = Path('/content/drive/MyDrive/논문/data')
M3_ROOT = ROOT/'results_v3_dunnhumby_clv_m3_centered_value_graph_v1'
if len(list(M3_ROOT.glob('clv_m3_centered_value_graph_*.json'))) != 1:
    raise RuntimeError(f'완료된 M1/M3 기준결과가 정확히 하나 필요합니다: {M3_ROOT}')
for seed in (42, 44):
    old = ROOT/f'results_v3_dunnhumby_clv_m5_m3_m4_user_centered_s{seed}_v1'
    if len(list(old.glob('clv-m5-m3-m4-user-centered-dev-v1_*.json'))) != 1:
        raise RuntimeError(f'완료된 seed{seed} M3+M4-C 기준결과가 필요합니다: {old}')

SOURCE_COMMIT = 'd1aa9642628aa92b90da3facdf5de9a4ab6ec33d'
REPO = Path('/content/clv-m5-history-m2-m3-m4-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
import clv_m5_history_m2_m3_m4_user_centered_screen as screen
print(json.dumps(screen.preflight_summary(), ensure_ascii=False, indent=2))

## 신규 arm 4회 학습

seed42의 두 arm을 완료한 뒤 seed44의 두 arm을 실행합니다. 런타임이 중단되면 같은 셀을 다시 실행하세요. 완료 arm은 재사용하고 미완료 arm은 저장된 epoch부터 재개합니다.

In [ ]:
result = screen.run()

In [ ]:
from IPython.display import display
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

metrics = ['recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50',
           'price_purchase_amount_weighted_hit@10','vndcg@10',
           'price_purchase_amount_weighted_hit@20','vndcg@20',
           'price_purchase_amount_weighted_hit@50','vndcg@50']
display(result['absolute'].query('epoch == 300')[['model_id','seed','epoch',*metrics]])
display(result['comparison'].query("epoch == 300 and contrast == 'FULL_minus_M3+M4-C' and metric in @metrics"))
display(result['factorial'].query('metric in @metrics'))
display(result['summary'])
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

zip_path = Path('/content/clv_m5_history_m2_m3_m4_user_centered_seed42_44_results.zip')
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, Path(path).name)
files.download(str(zip_path))